In [1]:
import polars as pl
import mappy as mp
import os
from collections import Counter
from collections import OrderedDict

In [4]:
input ='/home/n11702427/m/users/r_nurdiansyah/wagtail/testing/3_mappy/dna-sequences.fasta'
output_align='/home/n11702427/m/users/r_nurdiansyah/wagtail/testing/3_mappy/align'
output_meta='/home/n11702427/m/users/r_nurdiansyah/wagtail/testing/3_mappy/meta'
index='/home/n11702427/m/users/r_nurdiansyah/wagtail/0_database/danica.mmi'

In [3]:
#setting class so it can be used
#dataclass
class AmpliconDataset:
    input_file: str = None
    basename: str = None
    output_directory: str = None

In [3]:
#read index to mappy
ref = mp.Aligner(index, preset = 'sr')
if not ref: raise Exception("ERROR: failed to load/build index")
#print to console if index is loaded from the file

In [38]:
print(mp.__file__)

/mnt/hpccs01/work/microbiome/users/r_nurdiansyah/e/minimap-update/lib/python3.12/site-packages/mappy.cpython-312-x86_64-linux-gnu.so


In [5]:
#testing mappy to read fastq
for name, seq, qual in mp.fastx_read(input, read_comment=False):
    #print("{}\t{}\t{}".format(name, seq, qual))
    r1_name = name
    print(r1_name)

1c952ffa899157451e1a0803a6e7d1be
961fd2d00cf9d8631a7bf2e1e954678b
e2c0a94741782377cd3808323213edb7
9508d1e293eb48991dbcbed8ac8cf57d
d4c2eddd7f6efef8a159f05deecf7234
d5d03ff437157e429bf2ee3fe36082cb
b8b28a281d89e46e6e098250eb2690a8
3b3306459720de37604f4ac82d92fed8
fad117348b9cb3cb3f9b3a41f1341ffa
58b2da87846b2cf5f2002bf364e7cd38
774988575190ff17b5c62a5c91b0f574
d0fdfeb3138c01a6d3bdfc9575c24a75
ceb756736a42eac0b088578abde942b9
97f97975d309adf303f8248cc7cfc7a9
e4dac646715b26b2149e30e02581b3ca
b7a618cc2e2f6c22f533ce5ff8a86a9c
8a972df62756e85e331fa195231167f9
7722883efb81040117fcc5c24b74ab22
17109367a25b5799be057e9b4b079d8f
296b7b1fc7af0d48f3c4a5002b7ea839
c2705de3ba6a57e6776d68a10ae9e784
62021d6c403b0e03b1597ab090ad300d
121b6c54066c48c42f35313ae229e9d5
5f63590a17f64dba1c57a5b5ca30ac73
e98c584dbecf9c8de3d4cdc09f0ff091
096a6461bcb2745a2388b4044f52d5b1
8b8909155dd0cc40fa6b00c592c4578d
a7bd97c0939453694ec31e6e796c9e43
16b156381191fe61f7e34dc36198d633
c41e1bbde55fe9e7d60c035b9a2afde5
cce5a8ae81

In [6]:
# Initialize counters -> for metadata
total_mapq = 0
alignment_counter = Counter()

In [30]:
result_per_i = {}
for name, seq, qual in mp.fastx_read(input, read_comment=False):
    is_mapped = False #to track duplicates
    #for hits in ref.map(seq, cs=True, MD=True):
    for hits in ref.map(seq):
        if hits.is_primary:
            result_per_i[name] = (hits.ctg)
            alignment_counter['mapped'] += 1
            total_mapq += hits.mapq
            is_mapped = True
            break # Only consider primary alignment
        else:
            alignment_counter['unmapped'] += 1
    #count the number of hits using Counter
#ctg_counts = Counter(result_per_i.values())    
print(result_per_i)

{'1c952ffa899157451e1a0803a6e7d1be': 'FLASV871300.1370;tax=d:Bacteria,p:Firmicutes,c:Bacilli,o:Bacillales,f:Bacillaceae,g:Bacillus,s:MFD_s_24189;', '961fd2d00cf9d8631a7bf2e1e954678b': 'FLASV1017960.1381;tax=d:Bacteria,p:Firmicutes,c:Bacilli,o:Lactobacillales,f:Enterococcaceae,g:Enterococcus,s:MFD_s_1017960;', 'e2c0a94741782377cd3808323213edb7': 'FLASV832231.1372;tax=d:Bacteria,p:Firmicutes,c:Bacilli,o:Staphylococcales,f:Staphylococcaceae,g:Staphylococcus,s:MFD_s_126847;', '9508d1e293eb48991dbcbed8ac8cf57d': 'FLASV871300.1370;tax=d:Bacteria,p:Firmicutes,c:Bacilli,o:Bacillales,f:Bacillaceae,g:Bacillus,s:MFD_s_24189;', 'd4c2eddd7f6efef8a159f05deecf7234': 'FLASV1017960.1381;tax=d:Bacteria,p:Firmicutes,c:Bacilli,o:Lactobacillales,f:Enterococcaceae,g:Enterococcus,s:MFD_s_1017960;', 'd5d03ff437157e429bf2ee3fe36082cb': 'FLASV662431.1374;tax=d:Bacteria,p:Proteobacteria,c:Gammaproteobacteria,o:Pseudomonadales,f:Hahellaceae,g:Hahella,s:MFD_s_26319;', 'b8b28a281d89e46e6e098250eb2690a8': 'FLASV8322

In [8]:
# Calculate mapping percentage
total_reads = alignment_counter["mapped"] + alignment_counter["unmapped"]
mapping_percentage = (alignment_counter["mapped"] / total_reads) * 100 if total_reads > 0 else 0
average_mapq = total_mapq / alignment_counter["mapped"] if alignment_counter["mapped"] > 0 else 0

# Print the results
print(f"Total Reads: {total_reads}")
print(f"Mapped Reads: {alignment_counter['mapped']}")
print(f"Unmapped Reads: {alignment_counter['unmapped']}")
print(f"Mapping Percentage: {mapping_percentage:.2f}%")
print(f"Average MAPQ: {average_mapq:.2f}")

Total Reads: 129
Mapped Reads: 129
Unmapped Reads: 0
Mapping Percentage: 100.00%
Average MAPQ: 0.36


In [11]:
#prepare metadata into dataframe with column name: sample-id, total reads, mapped reads, unmapped reads, mapping percentage, average mapq
metadata = pl.DataFrame({
    'sample-id': ['V1V2'],
    'total reads': [total_reads],
    'mapped reads': [alignment_counter['mapped']],
    'unmapped reads': [alignment_counter['unmapped']],
    'mapping percentage': [mapping_percentage],
    'average mapq': [average_mapq]
})

In [12]:
print(metadata)

shape: (1, 6)
┌───────────┬─────────────┬──────────────┬────────────────┬────────────────────┬──────────────┐
│ sample-id ┆ total reads ┆ mapped reads ┆ unmapped reads ┆ mapping percentage ┆ average mapq │
│ ---       ┆ ---         ┆ ---          ┆ ---            ┆ ---                ┆ ---          │
│ str       ┆ i64         ┆ i64          ┆ i64            ┆ f64                ┆ f64          │
╞═══════════╪═════════════╪══════════════╪════════════════╪════════════════════╪══════════════╡
│ V1V2      ┆ 129         ┆ 129          ┆ 0              ┆ 100.0              ┆ 0.356589     │
└───────────┴─────────────┴──────────────┴────────────────┴────────────────────┴──────────────┘


In [42]:
# Prepare metadata dictionary
metadata = {
    "Total Reads": total_reads,
    "Mapped Reads": alignment_counter["mapped"],
    "Unmapped Reads": alignment_counter["unmapped"],
    "Mapping Percentage": f"{mapping_percentage:.2f}%",
    "Average Mapping Quality": f"{average_mapq:.2f}"
}

In [50]:
filename=(f"/home/n11702427/m/users/r_nurdiansyah/wagtail/testing/3_mappy/meta/{basename}_metadata.txt")
with open(filename, 'w') as f:
    for key, value in metadata.items():
        f.write(f"{key}: {value}\n")
    print(f"Metadata successfully saved to {filename}")

Metadata successfully saved to /home/n11702427/m/users/r_nurdiansyah/wagtail/testing/3_mappy/meta/ERR4994172_metadata.txt


In [44]:
print(alignment_counter["mapped"])

11136


In [13]:
print(ctg_counts)

Counter({'FLASV883688.1363;tax=d:Bacteria,p:Proteobacteria,c:Gammaproteobacteria,o:Enterobacterales,f:Enterobacteriaceae,g:Escherichia-Shigella,s:MFD_s_383;': 5, 'FLASV871300.1370;tax=d:Bacteria,p:Firmicutes,c:Bacilli,o:Bacillales,f:Bacillaceae,g:Bacillus,s:MFD_s_24189;': 4, 'FLASV1017960.1381;tax=d:Bacteria,p:Firmicutes,c:Bacilli,o:Lactobacillales,f:Enterococcaceae,g:Enterococcus,s:MFD_s_1017960;': 4, 'FLASV916992.1396;tax=d:Bacteria,p:Firmicutes,c:Bacilli,o:Lactobacillales,f:Lactobacillaceae,g:Limosilactobacillus,s:Lactobacillus_fermentum;': 4, 'FLASV884662.1370;tax=d:Bacteria,p:Firmicutes,c:Bacilli,o:Lactobacillales,f:Listeriaceae,g:Listeria,s:MFD_s_286000;': 4, 'FLASV414129.1396;tax=d:Bacteria,p:Firmicutes,c:Bacilli,o:Lactobacillales,f:Lactobacillaceae,g:Limosilactobacillus,s:Lactobacillus_fermentum;': 4, 'FLASV873342.1363;tax=d:Bacteria,p:Proteobacteria,c:Gammaproteobacteria,o:Enterobacterales,f:Enterobacteriaceae,g:Escherichia-Shigella,s:MFD_s_1065;': 4, 'FLASV825639.1396;tax=d:B

In [46]:
#get the base name of the input file
basename = os.path.basename(input).split('.')[0]
#remove the extension of the file
#basename = os.path.splitext(basename).split('.')[0]
print(basename)

ERR4994172


In [32]:
# Change the result per i into dataframe -> to match the table result
df = pl.from_dict(result_per_i)
df1 = df.transpose(include_header=True)
#result_per_i.columns = ['sample-id', 'ctg']
df1.columns = ['read', 'contig']
print(df1)

shape: (129, 2)
┌─────────────────────────────────┬─────────────────────────────────┐
│ read                            ┆ contig                          │
│ ---                             ┆ ---                             │
│ str                             ┆ str                             │
╞═════════════════════════════════╪═════════════════════════════════╡
│ 1c952ffa899157451e1a0803a6e7d1… ┆ FLASV871300.1370;tax=d:Bacteri… │
│ 961fd2d00cf9d8631a7bf2e1e95467… ┆ FLASV1017960.1381;tax=d:Bacter… │
│ e2c0a94741782377cd3808323213ed… ┆ FLASV832231.1372;tax=d:Bacteri… │
│ 9508d1e293eb48991dbcbed8ac8cf5… ┆ FLASV871300.1370;tax=d:Bacteri… │
│ d4c2eddd7f6efef8a159f05deecf72… ┆ FLASV1017960.1381;tax=d:Bacter… │
│ …                               ┆ …                               │
│ faa404f26355ade0452744f952b680… ┆ FLASV903375.1372;tax=d:Bacteri… │
│ 433fb165118a2fa968618a0b7d914a… ┆ FLASV1022410.1343;tax=d:Bacter… │
│ aa77e018552a2cc9f2e708c55c974a… ┆ FLASV981243.1363;tax=d:Bacteri… │
│ 39

In [23]:
df1.write_csv(f"/home/n11702427/m/users/r_nurdiansyah/wagtail/testing/3_mappy/align/V1V2_alignment.tsv", separator='\t')

In [26]:
test= pl.read_csv("/home/n11702427/m/users/r_nurdiansyah/wagtail/testing/3_mappy/align/V1V2_alignment.tsv", separator='\t', has_header=False)
test[:4]

column_1,column_2
str,str
"""read""","""contig"""
"""1c952ffa899157451e1a0803a6e7d1…","""FLASV871300.1370;tax=d:Bacteri…"
"""961fd2d00cf9d8631a7bf2e1e95467…","""FLASV1017960.1381;tax=d:Bacter…"
"""e2c0a94741782377cd3808323213ed…","""FLASV832231.1372;tax=d:Bacteri…"


In [47]:
#record the ctg_counts dictionary to a tsv file using polars
df1 = pl.from_dict(ctg_counts)
df1 = df1.transpose(include_header=True)
#add sample column on the first column and fill the value with user's input (sample name)
df1.columns = ['contig', 'count']
df1 = df1.with_columns(pl.Series("sample", [basename]*len(df1)))
#reorder the column to match the biobox script input while delete the unnecessary columns
df1 = df1[["sample", "contig", "count"]]
#sort based on count
df2 = df1.sort("count", descending=True)
df2[:4]

sample,contig,count
str,str,i64
"""ERR4994172""","""FLASV957672.13…",2875
"""ERR4994172""","""FLASV60.1391;t…",2148
"""ERR4994172""","""FLASV89.1392;t…",871
"""ERR4994172""","""FLASV927273.13…",592


In [49]:
df2.write_csv(f"/home/n11702427/m/users/r_nurdiansyah/wagtail/testing/3_mappy/align/{basename}_alignment.tsv", separator='\t')

In [18]:
for i in input_files:
    result_per_i = {}
    #Loop through single-end reads
    for name, seq, qual in mp.fastx_read(i):
        barcode = os.path.basename(i).split('.')[0]
        names = "{}_barcode+{}".format(barcode, name)
        #Perform single-end alignment
        for hits in ref.map(seq, cs=True, MD=True):
            if hits.is_primary:
                result_per_i[names] = (hits.ctg)
    #count the number of contigs using Counter
    ctg_counts = Counter(result_per_i.values())
print(ctg_counts)

Counter()


In [6]:
for i in input:
    result_per_i = {}
    #Loop through single-end reads
    for name, seq, qual in mp.fastx_read(i):
        barcode = os.path.basename(i).split('.')[0]
        names = "{}_barcode+{}".format(barcode, name)
        #Perform single-end alignment
        for hits in ref.map(seq, cs=True, MD=True):
            if hits.is_primary:
                result_per_i[names] = (hits.ctg)
    #count the number of contigs using Counter
    ctg_counts = Counter(result_per_i.values())
print(ctg_counts)

Counter()


In [9]:
# Check if input is a file or a directory and list all files in the directory
input_files=[]
printout=[]
#d = AmpliconDataset()
if isinstance (input, str):
    input = [input] #convert to list if input is a string
    
input_file = sorted(input_files)
file_names = [os.path.basename(f) for f in input_files]

d = AmpliconDataset()
datasets = []
for file in file_names:
    base = file.split('.')[0]
    d.basename = base
    datasets.append(d)

for input in input_file:
    d.input_file = input
    datasets.append(d)

print(d.input_file)
print(d.basename)

None
None


In [63]:
#making directory, no need to run this
for d in datasets:
    file = d.basename
    # Extract parent directory name (first 3 digits after "ERR")
    if len(file) < 6:
        logging.error(f"Filename {file} is too short")
        sys.exit(1)
    parent_dir = file[:6]
    
    # Extract the last digit to determine the subdirectory ("000" to "009")
    sub_dir_num = int(file[-3:]) % 1000
    
    # Create the full path for the accession directory
    new_dir = os.path.join(output, f"{parent_dir}", f"{sub_dir_num}", file)
    d.output_directory = new_dir
          
    #Create new directory if not exist
    if not os.path.exists(new_dir):
        os.makedirs(new_dir, exist_ok=True)
        logging.info(f"Created directory: {new_dir}")
    else:
        logging.info(f"Directory already exists: {new_dir}")

In [12]:
# Create empty lists to store alignment data and metadata counts
alignments = []
has_primary_flags = {}

In [20]:
for i in input:
    result_per_i = {}
    #Loop through single-end reads
    for name, seq, qual in mp.fastx_read(i):
        barcode = os.path.basename(i).split('.')[0]
        names = "{}_barcode+{}".format(barcode, name)
        #Perform single-end alignment
        for hits in ref.map(seq, cs=True, MD=True):
            if hits.is_primary:
                result_per_i[names] = (hits.ctg)
    #count the number of contigs using Counter
    ctg_counts = Counter(result_per_i.values())
print(ctg_counts)

Counter()


In [18]:
# Process each input sequence in the input file (assuming FASTA/FASTQ format)
for i in input:
    result_per_i = {}
    for name, seq, qual in mp.fastx_read(i):
        alignment_counts['total'] += 1
        #has_primary = False  # To track duplicates

        for hits in ref.map(seq, cs=True, MD=True):
            if hits.is_primary:
                result_per_i[names] = (hits.ctg)
        #count the number of contigs using Counter
    ctg_counts = Counter(result_per_i.values())
print(ctg_counts)
        # # Perform alignment and store results
        # for hit in aligner.map(seq):
        #     # Store alignment result in list
        #     alignments.append([name, hit.ctg])

        #     if hit.is_primary and not has_primary:
        #         alignment_counts['primary'] += 1
        #         has_primary = True
        #     elif hit.is_secondary:
        #         alignment_counts['secondary'] += 1
        #     if hit.is_supplementary:
        #         alignment_counts['supplementary'] += 1
        #     if has_primary and hit.is_primary:  # Mark duplicates
        #         alignment_counts['duplicates'] += 1

print(alignments)
    # Convert alignment list to Polars DataFrame and save as TSV
    # alignment_df = pl.DataFrame(alignments, schema=["read_name", "ctg"])
    # alignment_df.write_csv(alignment_output, separator="\t", has_header=True)

Counter()
[]


In [53]:
for da in datasets:
    i = da.input_file
    d = da.output_directory

    print("Running analysis for {}".format(i))
    result_per_i = {}
    ctg_counts = Counter()
    #Loop through single-end reads
    for name, seq, qual in mp.fastx_read(i):
        barcode = os.path.basename(i).split('.')[0]
        names = "{}_barcode+{}".format(barcode, name)
        #Perform single-end alignment
        for hits in ref.map(seq, cs=True, MD=True):
            if hits.is_primary:
                result_per_i[names] = (hits.ctg)
        #count the number of contigs using Counter
        ctg_counts.update(result_per_i.values())
        print(ctg.counts)

Running analysis for V3V4005.fastq.gz
Running analysis for V1V2002.fastq.gz
Running analysis for V1V2001.fastq.gz
Running analysis for V4V6007.fastq.gz
Running analysis for V4V6008.fastq.gz
Running analysis for V1V3004.fastq.gz
Running analysis for V3V4006.fastq.gz
Running analysis for V1V3003.fastq.gz


In [36]:
help(mp)

Help on module mappy:

NAME
    mappy

CLASSES
    builtins.object
        Aligner
        Alignment
        ThreadBuffer

    class Aligner(builtins.object)
     |  Methods defined here:
     |
     |  __bool__(self, /)
     |      True if self else False
     |
     |  __reduce__ = __reduce_cython__(...)
     |
     |  __setstate__ = __setstate_cython__(...)
     |
     |  map(self, seq, seq2=None, buf=None, cs=False, MD=False, max_frag_len=None, extra_flags=None)
     |
     |  seq(self, name, start=0, end=2147483647)
     |
     |  ----------------------------------------------------------------------
     |  Static methods defined here:
     |
     |  __new__(*args, **kwargs)
     |      Create and return a new object.  See help(type) for accurate signature.
     |
     |  ----------------------------------------------------------------------
     |  Data descriptors defined here:
     |
     |  k
     |
     |  n_seq
     |
     |  seq_names
     |
     |  w

    class Alignment(